In [1]:
import sys
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DIR
from src.utils import preview
from src.features import transit_tier

In [2]:
import sys
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DIR
from src.utils import preview

In [3]:
nyc_model_ready_gdf = gpd.read_file(PROCESSED_DIR / "nyc_model_ready.geojson")
preview(nyc_model_ready_gdf, "NYC Model Ready")
nyc_model_ready_gdf.head()

NYC Model Ready: 206 rows x 8 cols
  zip_code   boroname  total_population  median_household_income  \
0    10028  Manhattan             45679                 168125.0   
1    11427     Queens             25124                  92129.0   
2    11697     Queens              4123                 134844.0   
3    11004     Queens             14296                 109865.0   
4    10282  Manhattan              5960                 250001.0   

   median_household_income_filled  nearest_tj_distance_miles  \
0                        168125.0                   1.231225   
1                         92129.0                   5.405433   
2                        134844.0                  10.037851   
3                        109865.0                   7.351747   
4                        250001.0                   0.836176   

   subway_station_count                                           geometry  
0                   2.0  POLYGON ((994561.592 222839.437, 994690.277 22...  
1                

,zip_code,boroname,total_population,median_household_income,median_household_income_filled,nearest_tj_distance_miles,subway_station_count,geometry
0,10028,Manhattan,45679,168125.0,168125.0,1.231225,2.0,"POLYGON ((994561.592 222839.437, 994690.277 22..."
1,11427,Queens,25124,92129.0,92129.0,5.405433,0.0,"POLYGON ((1049040.311 204011.201, 1049258.374 ..."
2,11697,Queens,4123,134844.0,134844.0,10.037851,0.0,"POLYGON ((1000721.481 136681.744, 1000888.65 1..."
3,11004,Queens,14296,109865.0,109865.0,7.351747,0.0,"POLYGON ((1061069.68 214228.676, 1061117.367 2..."
4,10282,Manhattan,5960,250001.0,250001.0,0.836176,0.0,"POLYGON ((979421.96 199752.189, 979617.937 201..."


In [4]:
# make sure we're in a projected CRS with feet
nyc_model_ready_gdf = nyc_model_ready_gdf.to_crs(epsg=2263)

nyc_model_ready_gdf["area_sq_miles"] = nyc_model_ready_gdf.geometry.area / (5280 ** 2)

nyc_model_ready_gdf[["zip_code", "area_sq_miles"]].head()

,zip_code,area_sq_miles
0,10028,0.315510
1,11427,1.568670
2,11697,2.047392
3,11004,0.951062
4,10282,0.068667


In [5]:
nyc_model_ready_gdf["population_density"] = (
    nyc_model_ready_gdf["total_population"] / nyc_model_ready_gdf["area_sq_miles"]
)

nyc_model_ready_gdf[[
    "zip_code",
    "total_population",
    "area_sq_miles",
    "population_density"
]].head()

,zip_code,total_population,area_sq_miles,population_density
0,10028,45679,0.315510,144778.293180
1,11427,25124,1.568670,16016.113999
2,11697,4123,2.047392,2013.781402
3,11004,14296,0.951062,15031.623052
4,10282,5960,0.068667,86795.712887


In [6]:
nyc_model_ready_gdf["income_percentile"] = nyc_model_ready_gdf[
    "median_household_income_filled"
].rank(pct=True)

In [7]:
nyc_model_ready_gdf[[
    "zip_code",
    "median_household_income_filled",
    "income_percentile"
]].head()

,zip_code,median_household_income_filled,income_percentile
0,10028,168125.0,0.941748
1,11427,92129.0,0.631068
2,11697,134844.0,0.868932
3,11004,109865.0,0.771845
4,10282,250001.0,0.995146


In [8]:
nyc_model_ready_gdf["transit_access_tier"] = nyc_model_ready_gdf[
    "subway_station_count"
].apply(transit_tier)

In [9]:
nyc_model_ready_gdf["transit_access_tier"].value_counts()

transit_access_tier
low       95
medium    73
high      38
Name: count, dtype: int64

In [10]:
nyc_model_ready_gdf["is_tj_gap"] = (
    nyc_model_ready_gdf["nearest_tj_distance_miles"] >= 3
).astype(int)

In [11]:
nyc_model_ready_gdf["is_tj_gap"].value_counts()

is_tj_gap
0    123
1     83
Name: count, dtype: int64

In [12]:
nyc_model_ready_gdf["log_population_density"] = np.log1p(
    nyc_model_ready_gdf["population_density"]
)

In [13]:
print("shape:", nyc_model_ready_gdf.shape)
print("duplicate zip_code:", nyc_model_ready_gdf["zip_code"].duplicated().sum())
print("missing area_sq_miles:", nyc_model_ready_gdf["area_sq_miles"].isna().sum())
print("missing population_density:", nyc_model_ready_gdf["population_density"].isna().sum())
print("missing income_percentile:", nyc_model_ready_gdf["income_percentile"].isna().sum())
print("missing transit_access_tier:", nyc_model_ready_gdf["transit_access_tier"].isna().sum())
print("missing is_tj_gap:", nyc_model_ready_gdf["is_tj_gap"].isna().sum())
print("missing log_population_density:", nyc_model_ready_gdf["log_population_density"].isna().sum())

shape: (206, 14)
duplicate zip_code: 0
missing area_sq_miles: 0
missing population_density: 0
missing income_percentile: 0
missing transit_access_tier: 0
missing is_tj_gap: 0
missing log_population_density: 0


In [14]:
nyc_model_ready_gdf[[
    "zip_code",
    "boroname",
    "area_sq_miles",
    "population_density",
    "log_population_density",
    "income_percentile",
    "subway_station_count",
    "transit_access_tier",
    "nearest_tj_distance_miles",
    "is_tj_gap"
]].head(10)

,zip_code,boroname,area_sq_miles,population_density,log_population_density,income_percentile,subway_station_count,transit_access_tier,nearest_tj_distance_miles,is_tj_gap
0,10028,Manhattan,0.315510,144778.293180,11.882966,0.941748,2.0,medium,1.231225,0
1,11427,Queens,1.568670,16016.113999,9.681413,0.631068,0.0,low,5.405433,1
2,11697,Queens,2.047392,2013.781402,7.608266,0.868932,0.0,low,10.037851,1
3,11004,Queens,0.951062,15031.623052,9.617978,0.771845,0.0,low,7.351747,1
4,10282,Manhattan,0.068667,86795.712887,11.371324,0.995146,0.0,low,0.836176,0
5,10027,Manhattan,0.942827,69013.726017,11.142075,0.199029,5.0,high,0.508789,0
6,11432,Queens,2.023413,31496.289456,10.357657,0.281553,3.0,medium,2.944129,0
7,10452,Bronx,0.982204,78135.526762,11.266213,0.038835,5.0,high,2.369266,0
8,10469,Bronx,2.425794,28796.761977,10.268053,0.296117,3.0,medium,6.671917,1
9,11225,Brooklyn,0.874835,64633.901445,11.076510,0.393204,7.0,high,2.597065,0


In [15]:
nyc_model_ready_gdf.to_file(
    PROCESSED_DIR / "nyc_model_features.geojson",
    driver="GeoJSON"
)

nyc_model_ready_gdf.drop(columns=["geometry"], errors="ignore").to_csv(
    PROCESSED_DIR / "nyc_model_features.csv",
    index=False
)

# Summary
1. Created the core engineered ZIP-level features used in the model, including land area, population density, and log population density.
2. Converted demographic and access variables into more comparable business-facing features such as income_percentile and transit_access_tier.
3. Measured whitespace from existing Trader Joe’s stores and created variables that capture spacing and candidate-market attractiveness.
4. Exported the final engineered feature table as both CSV and GeoJSON for use in the scoring notebook.